In [1]:
# Download and split dataset into train and validation folders

import os
import pathlib
import random
import shutil
import tensorflow as tf

# ----- Fixed project paths -----
base_dir = pathlib.Path(r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-2_Transfer_Learning")
dataset_root = base_dir / "dataset"
train_dir = dataset_root / "train"
val_dir = dataset_root / "validation"

train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

# ----- Download + extract dataset -----
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive_path = tf.keras.utils.get_file(
    fname="flower_photos.tgz",
    origin=url,
    extract=True
)

# ----- Locate extracted folder (Windows-safe) -----
cache_root = pathlib.Path(archive_path).parent

source_dir = None
for root, dirs, files in os.walk(cache_root):
    if "flower_photos" in dirs:
        source_dir = pathlib.Path(root) / "flower_photos"
        break

if source_dir is None:
    raise FileNotFoundError(
        "flower_photos folder not found after extraction.\n"
        f"Cache root searched: {cache_root}"
    )

print("✅ Source dataset found at:", source_dir)

# ----- Split into train / validation -----
random.seed(42)
split_ratio = 0.8

classes = sorted([d for d in source_dir.iterdir() if d.is_dir()], key=lambda x: x.name)

# Clean existing folders
for target in [train_dir, val_dir]:
    for item in target.iterdir():
        if item.is_dir():
            shutil.rmtree(item)

for cls in classes:
    images = [p for p in cls.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    random.shuffle(images)

    n_train = int(len(images) * split_ratio)
    train_images = images[:n_train]
    val_images = images[n_train:]

    (train_dir / cls.name).mkdir(parents=True, exist_ok=True)
    (val_dir / cls.name).mkdir(parents=True, exist_ok=True)

    for img in train_images:
        shutil.copy2(img, train_dir / cls.name / img.name)

    for img in val_images:
        shutil.copy2(img, val_dir / cls.name / img.name)

print("✅ Dataset split complete")

# ----- Verify counts -----
def count_images(folder):
    return {
        d.name: len([f for f in d.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])
        for d in sorted(folder.iterdir()) if d.is_dir()
    }

print("Train counts:", count_images(train_dir))
print("Validation counts:", count_images(val_dir))

✅ Source dataset found at: C:\Users\hasan\.keras\datasets\flower_photos_extracted\flower_photos
✅ Dataset split complete
Train counts: {'daisy': 506, 'dandelion': 718, 'roses': 512, 'sunflowers': 559, 'tulips': 639}
Validation counts: {'daisy': 127, 'dandelion': 180, 'roses': 129, 'sunflowers': 140, 'tulips': 160}


In [2]:
# Load dataset with ImageDataGenerator (train/validation) and basic settings

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_data_dir = r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-2_Transfer_Learning\dataset\train"
val_data_dir   = r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-2_Transfer_Learning\dataset\validation"

# NOTE: Use only augmentation here. No rescale. We'll use model preprocess in the model pipeline later if needed.
train_datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator()

train_gen = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    val_data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES, CLASS_NAMES


Found 2934 images belonging to 5 classes.
Found 736 images belonging to 5 classes.


(5, ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips'])

In [3]:
# Utility: build transfer learning model (frozen or fine-tuned)

from tensorflow.keras import layers, models

def build_tl_model(base_model, num_classes: int, preprocess_layer, fine_tune_at=None, lr=1e-3):
    """
    base_model: Keras Applications model with include_top=False
    preprocess_layer: a Keras layer that applies model-specific preprocessing
    fine_tune_at:
      - None => fully frozen base model (feature extractor)
      - int  => unfreeze layers from this index onward
    """
    base_model.trainable = False

    if fine_tune_at is not None:
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False
        for layer in base_model.layers[fine_tune_at:]:
            layer.trainable = True

    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = preprocess_layer(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [4]:
# Train + evaluate helper (collect val accuracy and val loss)

def train_and_eval(model, train_gen, val_gen, epochs=3):
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        verbose=1
    )
    best_val_acc = float(np.max(history.history["val_accuracy"]))
    best_val_loss = float(np.min(history.history["val_loss"]))
    return best_val_acc, best_val_loss, history


In [5]:
# Experiment 1: MobileNetV2 (frozen vs fine-tuned)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre

results = []

preprocess_layer = tf.keras.layers.Lambda(mob_pre)

# Frozen
base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
model_frozen = build_tl_model(base, NUM_CLASSES, preprocess_layer, fine_tune_at=None, lr=1e-3)
acc, loss, _ = train_and_eval(model_frozen, train_gen, val_gen, epochs=3)
results.append({"model":"MobileNetV2", "mode":"Frozen", "best_val_acc":acc, "best_val_loss":loss})

# Fine-tune (unfreeze last ~30 layers)
base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
fine_tune_at = max(len(base.layers) - 30, 0)
model_ft = build_tl_model(base, NUM_CLASSES, preprocess_layer, fine_tune_at=fine_tune_at, lr=1e-5)
acc, loss, _ = train_and_eval(model_ft, train_gen, val_gen, epochs=3)
results.append({"model":"MobileNetV2", "mode":"Fine-tuned", "best_val_acc":acc, "best_val_loss":loss})

pd.DataFrame(results)


Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 151s 788ms/step - accuracy: 0.7147 - loss: 0.7737 - val_accuracy: 0.8438 - val_loss: 0.4316
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 96s 522ms/step - accuracy: 0.8361 - loss: 0.4494 - val_accuracy: 0.8682 - val_loss: 0.3640
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 97s 527ms/step - accuracy: 0.8650 - loss: 0.3736 - val_accuracy: 0.8696 - val_loss: 0.3575
Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 131s 652ms/step - accuracy: 0.4516 - loss: 1.3545 - val_accuracy: 0.6318 - val_loss: 0.9669
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 119s 647ms/step - accuracy: 0.7168 - loss: 0.7784 - val_accuracy: 0.7717 - val_loss: 0.6515
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 115s 622ms/step - accuracy: 0.7894 - loss: 0.6083 - val_accuracy: 0.8179 - val_loss: 0.5124


,model,mode,best_val_acc,best_val_loss
0,MobileNetV2,Frozen,0.869565,0.357493
1,MobileNetV2,Fine-tuned,0.817935,0.512362


In [6]:
# Experiment 2: ResNet50 (frozen vs fine-tuned)

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre

preprocess_layer = tf.keras.layers.Lambda(res_pre)

# Frozen
base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
model_frozen = build_tl_model(base, NUM_CLASSES, preprocess_layer, fine_tune_at=None, lr=1e-3)
acc, loss, _ = train_and_eval(model_frozen, train_gen, val_gen, epochs=3)
results.append({"model":"ResNet50", "mode":"Frozen", "best_val_acc":acc, "best_val_loss":loss})

# Fine-tune (unfreeze last ~40 layers)
base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
fine_tune_at = max(len(base.layers) - 40, 0)
model_ft = build_tl_model(base, NUM_CLASSES, preprocess_layer, fine_tune_at=fine_tune_at, lr=1e-5)
acc, loss, _ = train_and_eval(model_ft, train_gen, val_gen, epochs=3)
results.append({"model":"ResNet50", "mode":"Fine-tuned", "best_val_acc":acc, "best_val_loss":loss})

pd.DataFrame(results)


Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 349s 2s/step - accuracy: 0.7440 - loss: 0.6859 - val_accuracy: 0.8587 - val_loss: 0.3689
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 318s 2s/step - accuracy: 0.8684 - loss: 0.3751 - val_accuracy: 0.9062 - val_loss: 0.2738
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 316s 2s/step - accuracy: 0.8916 - loss: 0.3072 - val_accuracy: 0.9076 - val_loss: 0.2789
Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 473s 2s/step - accuracy: 0.5988 - loss: 1.0443 - val_accuracy: 0.8234 - val_loss: 0.4855
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 460s 2s/step - accuracy: 0.8282 - loss: 0.5121 - val_accuracy: 0.8804 - val_loss: 0.3431
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 456s 2s/step - accuracy: 0.8630 - loss: 0.3812 - val_accuracy: 0.9062 - val_loss: 0.2785


,model,mode,best_val_acc,best_val_loss
0,MobileNetV2,Frozen,0.869565,0.357493
1,MobileNetV2,Fine-tuned,0.817935,0.512362
2,ResNet50,Frozen,0.907609,0.273838
3,ResNet50,Fine-tuned,0.906250,0.278496


In [7]:
# Experiment 3: EfficientNetB0 (frozen vs fine-tuned)

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre

preprocess_layer = tf.keras.layers.Lambda(eff_pre)

# Frozen
base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
model_frozen = build_tl_model(base, NUM_CLASSES, preprocess_layer, fine_tune_at=None, lr=1e-3)
acc, loss, _ = train_and_eval(model_frozen, train_gen, val_gen, epochs=3)
results.append({"model":"EfficientNetB0", "mode":"Frozen", "best_val_acc":acc, "best_val_loss":loss})

# Fine-tune (unfreeze last ~30 layers)
base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
fine_tune_at = max(len(base.layers) - 30, 0)
model_ft = build_tl_model(base, NUM_CLASSES, preprocess_layer, fine_tune_at=fine_tune_at, lr=1e-5)
acc, loss, _ = train_and_eval(model_ft, train_gen, val_gen, epochs=3)
results.append({"model":"EfficientNetB0", "mode":"Fine-tuned", "best_val_acc":acc, "best_val_loss":loss})

pd.DataFrame(results)


Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 132s 653ms/step - accuracy: 0.7389 - loss: 0.7395 - val_accuracy: 0.8859 - val_loss: 0.3843
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 118s 640ms/step - accuracy: 0.8742 - loss: 0.3954 - val_accuracy: 0.8981 - val_loss: 0.2970
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 117s 633ms/step - accuracy: 0.8920 - loss: 0.3276 - val_accuracy: 0.9076 - val_loss: 0.2627
Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 153s 747ms/step - accuracy: 0.3630 - loss: 1.4854 - val_accuracy: 0.6019 - val_loss: 1.1937
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 134s 728ms/step - accuracy: 0.6080 - loss: 1.1779 - val_accuracy: 0.7636 - val_loss: 0.9068
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 134s 726ms/step - accuracy: 0.7140 - loss: 0.9585 - val_accuracy: 0.8288 - val_loss: 0.7119


,model,mode,best_val_acc,best_val_loss
0,MobileNetV2,Frozen,0.869565,0.357493
1,MobileNetV2,Fine-tuned,0.817935,0.512362
2,ResNet50,Frozen,0.907609,0.273838
3,ResNet50,Fine-tuned,0.906250,0.278496
4,EfficientNetB0,Frozen,0.907609,0.262727
5,EfficientNetB0,Fine-tuned,0.828804,0.711887


In [8]:
# Final results table (sorted by best_val_acc)

df_task4_2 = pd.DataFrame(results)
df_task4_2_sorted = df_task4_2.sort_values(by="best_val_acc", ascending=False).reset_index(drop=True)
df_task4_2_sorted

,model,mode,best_val_acc,best_val_loss
0,EfficientNetB0,Frozen,0.907609,0.262727
1,ResNet50,Frozen,0.907609,0.273838
2,ResNet50,Fine-tuned,0.906250,0.278496
3,MobileNetV2,Frozen,0.869565,0.357493
4,EfficientNetB0,Fine-tuned,0.828804,0.711887
5,MobileNetV2,Fine-tuned,0.817935,0.512362
